# Test des VAE équivariants (C4 ou SO(2))

Même tests que test_vae.ipynb sur un VAE équivariant aux rotations, comparé au
VAE convolutif (mnist_vae.py). `MODEL` choisit le VAE testé :
- `"c4"` : équivariant aux rotations de 90° (mnist_vae_equivariant.py) ;
- `"so2"` : équivariant à toutes les rotations (mnist_vae_so2.py), exactement
  pour les angles droits et approximativement pour les autres.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from numb_project.constants import EQUIVARIANT_VAE_CHECKPOINT_FILE, SO2_VAE_CHECKPOINT_FILE, VAE_CHECKPOINT_FILE
from numb_project.mnist_vae_equivariant import EquivariantVAE
from numb_project.mnist_vae_so2 import SO2VAE
from numb_project.notebook_utils import *

MODEL = "so2"  # "c4" ou "so2"

torch.set_grad_enabled(False)  # notebook d'évaluation uniquement

VAE_CLS, CHECKPOINT, NAME = {
    "c4": (EquivariantVAE, EQUIVARIANT_VAE_CHECKPOINT_FILE, "C4"),
    "so2": (SO2VAE, SO2_VAE_CHECKPOINT_FILE, "SO(2)"),
}[MODEL]
# Angles auxquels le latent se transforme de façon connue (rotate_latent)
EQUIVARIANCE_ANGLES = [90, 180, 270] if MODEL == "c4" else [15, 30, 45, 90, 135, 180, 270]

vae = load_vae(CHECKPOINT, VAE_CLS)
conv_vae = load_vae(VAE_CHECKPOINT_FILE)
images, labels = mnist_batch(64)


def encode(x: torch.Tensor, batch_size: int = 1000) -> tuple[torch.Tensor, torch.Tensor]:
    """(mu, logvar) de `vae` par lots : le VAE SO(2) ne tient pas en mémoire
    GPU sur 10 000 images d'un coup."""
    mus, logvars = zip(*(vae.encoder(batch) for batch in x.split(batch_size)))
    return torch.cat(mus), torch.cat(logvars)

# Disque central : rotate_images met les coins à 0, on compare les images dedans
_coords = torch.arange(28, device=images.device) - 13.5
disk = (_coords[:, None] ** 2 + _coords[None, :] ** 2 < 13 ** 2).float()

In [ ]:
# --- Vérification de l'équivariance sur des images réelles ---
# encoder(rotation(x)) doit valoir rotate_latent(encoder(x)) : pour C4, décalage
# circulaire des 4 coordonnées de chaque champ régulier ; pour SO(2), scalaires
# inchangés et vecteurs tournés du même angle. Le décodeur doit commuter avec
# la rotation. Erreurs relatives (norme), images comparées dans le disque
# central. Pour le VAE convolutif, aucune structure de ce type n'est attendue.
def relative_error(a: torch.Tensor, b: torch.Tensor) -> float:
    return ((a - b).norm() / b.norm()).item()

mu = vae.encoder(images)[0]
recon, conv_recon = vae.decoder(mu) * disk, vae_reconstruct(conv_vae, images) * disk
for angle in EQUIVARIANCE_ANGLES:
    rotated = rotate_by(images, angle)
    mu_rot = vae.encoder(rotated)[0]
    latent_err = relative_error(mu_rot, vae.rotate_latent(mu, angle))
    decoder_err = relative_error(vae.decoder(vae.rotate_latent(mu, angle)) * disk, rotate_by(recon, angle))
    conv_err = relative_error(vae_reconstruct(conv_vae, rotated) * disk, rotate_by(conv_recon, angle))
    print(f"rotation {angle:>3}° : latent {NAME} -> erreur {latent_err:.1e}, "
          f"décodage du latent tourné vs reconstruction tournée {decoder_err:.1e} "
          f"(VAE convolutif, reconstructions : {conv_err:.1e})")

In [ ]:
# --- MSE de reconstruction en fonction de l'angle de rotation de l'entrée ---
# Les angles droits (0, 90, 180, 270°) ont la même MSE pour un VAE équivariant ;
# pour SO(2), la courbe doit être presque plate sur tous les angles.
angles = list(range(0, 360, 15))
mse = {f"{NAME} équivariant": [], "convolutif": []}
for angle in angles:
    rotated = rotate_by(images, angle)
    mse[f"{NAME} équivariant"].append(F.mse_loss(vae_reconstruct(vae, rotated), rotated).item())
    mse["convolutif"].append(F.mse_loss(vae_reconstruct(conv_vae, rotated), rotated).item())
    print(f"angle={angle:>3}°: MSE {NAME} = {mse[f'{NAME} équivariant'][-1]:.5f}   MSE convolutif = {mse['convolutif'][-1]:.5f}")

plt.figure(figsize=(8, 4))
for name, values in mse.items():
    plt.plot(angles, values, marker="o", label=name)
for right_angle in (90, 180, 270):
    plt.axvline(right_angle, color="gray", linestyle=":", linewidth=1)
plt.xlabel("Angle de rotation de l'entrée (degrés)")
plt.ylabel("MSE de reconstruction")
plt.title("Qualité de reconstruction selon l'angle d'entrée")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- Comparaison qualitative à quelques angles (MSE par image en titre) ---
rows = []
for angle in [0, 45, 90, 135, 180, 270]:
    rotated = rotate_by(images, angle)
    rows.append((f"entrée {angle}°", rotated))
    for name, model in [(NAME, vae), ("convolutif", conv_vae)]:
        recon = vae_reconstruct(model, rotated)
        mse_per_image = ((recon - rotated) ** 2).mean(dim=(1, 2, 3))
        rows.append((name, recon, [f"{m:.3f}" for m in mse_per_image.tolist()]))
show_rows(rows, n_show=6)

In [ ]:
# --- Dimensions actives de l'espace latent ---
# Une dimension est "active" si son mu varie selon l'image (Var_x[mu_d] > 0.01).
# Une dimension morte a un mu ~ constant et un logvar ~ 0 (posterior = prior) :
# elle ne porte aucune information, sa KL moyenne est ~ 0.
# Pour SO(2), les dimensions 2k et 2k+1 de la partie vectorielle sont les deux
# coordonnées d'un même vecteur et ont la même statistique sur des angles aléatoires.
all_images, _ = mnist_batch(10000)
mu, logvar = encode(rotate_images(all_images, random_angles(len(all_images))))

mu_var = mu.var(dim=0).cpu()
kl_per_dim = (-0.5 * (1 + logvar - mu.pow(2) - logvar.exp())).mean(dim=0).cpu()

threshold = 0.01
active = mu_var > threshold
print(f"Dimensions actives : {active.sum().item()} / {len(mu_var)}")
for d in torch.argsort(mu_var, descending=True).tolist():
    print(f"dim {d:>2}: Var[mu]={mu_var[d]:.4f}  KL={kl_per_dim[d]:.3f}  {'actif' if active[d] else 'mort'}")

order = torch.argsort(mu_var, descending=True)
colors = ["C0" if a else "C7" for a in active[order]]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(mu_var)), mu_var[order], color=colors)
axes[0].axhline(threshold, color="red", linestyle="--", label=f"seuil {threshold}")
axes[0].set_yscale("log")
axes[0].set_ylabel("Var[mu] sur le test set")
axes[0].legend()
axes[1].bar(range(len(kl_per_dim)), kl_per_dim[order], color=colors)
axes[1].set_ylabel("KL moyenne (nats)")
for ax in axes:
    ax.set_xticks(range(len(mu_var)), order.tolist())
    ax.set_xlabel("dimension (triée)")
plt.tight_layout()
plt.show()

In [ ]:
# --- Sonde d'angle : l'angle de rotation est-il encodé dans mu ? ---
# On apprend à prédire (cos, sin) de l'angle à partir de mu, sur le train set,
# et on évalue sur le test set :
#  - sonde linéaire : l'angle est-il lisible "directement" dans mu ?
#  - sonde MLP      : l'angle est-il présent, même de façon non linéaire ?
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor

def encode_rotated(train: bool, n_images: int, seed: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    x, x_labels = mnist_batch(n_images, train=train, seed=seed)
    x_angles = random_angles(n_images, seed=seed)
    return encode(rotate_images(x, x_angles))[0].cpu(), x_angles.cpu(), x_labels.cpu()

mu_train, angle_train, _ = encode_rotated(train=True, n_images=20000, seed=0)
mu_test, angle_test, label_test = encode_rotated(train=False, n_images=10000, seed=1)

probes = {
    "linéaire": Ridge(alpha=1.0),
    "MLP": MLPRegressor(hidden_layer_sizes=(256, 256), max_iter=200, early_stopping=True, random_state=0),
}
pred_angles, errors = {}, {}
for name, probe in probes.items():
    probe.fit(mu_train.numpy(), angle_to_cos_sin(angle_train).numpy())
    pred_angles[name] = cos_sin_to_angle(torch.from_numpy(probe.predict(mu_test.numpy())))
    errors[name] = angle_error(pred_angles[name], angle_test)
    print(f"sonde {name:<8}: erreur moyenne = {errors[name].mean():5.1f}°, médiane = {errors[name].median():5.1f}°, "
          f"< 15° pour {(errors[name] < 15).float().mean():.1%} des images")
print("(hasard : erreur moyenne de 90°)")

# Erreur par chiffre : 0, 1 et 8 sont presque symétriques par rotation de 180°,
# leur angle peut être ambigu même pour un encodeur parfait.
print("\nErreur médiane par chiffre :")
for digit in range(10):
    mask = label_test == digit
    print(f"  {digit}: " + "  ".join(f"{name} {err[mask].median():5.1f}°" for name, err in errors.items()))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name in zip(axes[:2], probes):
    ax.scatter(angle_test, pred_angles[name], s=1, alpha=0.2)
    ax.plot([0, 360], [0, 360], color="red", linewidth=1)
    ax.set_xlabel("angle réel (°)")
    ax.set_ylabel("angle prédit (°)")
    ax.set_title(f"sonde {name} (erreur médiane {errors[name].median():.1f}°)")
width = 0.4
for i, (name, err) in enumerate(errors.items()):
    axes[2].bar(torch.arange(10) + (i - 0.5) * width, [err[label_test == d].median() for d in range(10)], width, label=name)
axes[2].set_xticks(range(10))
axes[2].set_xlabel("chiffre")
axes[2].set_ylabel("erreur médiane (°)")
axes[2].legend()
plt.tight_layout()
plt.show()